# LA Studio voice-isolation — UVR MDX-Net Vocals FT

This notebook loads exactly `sherpa-onnx-uvr-vocals-ft` (`k2-fsa/sherpa-onnx-uvr-vocals-ft`) on CUDA.
It does not use API Gateway and refuses every other model ID.

1. Choose **Runtime → Change runtime type → GPU**.
2. Run all cells.
3. Copy the printed URL and token into LA Studio.


In [ ]:
!nvidia-smi
%pip install -q "sherpa-onnx==1.13.4+cuda12.cudnn9" --find-links https://k2-fsa.github.io/sherpa/onnx/cuda.html
%pip install -q "soundfile==0.13.1" "fastapi==0.115.12" "uvicorn==0.34.3" "python-multipart==0.0.20"

!wget -q --show-progress -O /content/UVR-MDX-NET-Voc_FT.onnx https://github.com/k2-fsa/sherpa-onnx/releases/download/source-separation-models/UVR-MDX-NET-Voc_FT.onnx


In [ ]:
from pathlib import Path

WORKER = Path('/content/la_studio_separation_worker.py')
WORKER.write_text('import os\nimport secrets\nimport shutil\nimport subprocess\nimport threading\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport sherpa_onnx\nimport soundfile as sf\nfrom fastapi import FastAPI, File, Form, Header, HTTPException, UploadFile\nfrom fastapi.responses import FileResponse\n\nif "+cuda" not in sherpa_onnx.__version__:\n    raise RuntimeError("The installed sherpa-onnx wheel is not CUDA-enabled")\n\nMODEL_ID = "sherpa-onnx-uvr-vocals-ft"\nMODEL_NAME = "UVR MDX-Net Vocals FT"\nUPSTREAM_MODEL = "k2-fsa/sherpa-onnx-uvr-vocals-ft"\nARTIFACT_URL = "https://github.com/k2-fsa/sherpa-onnx/releases/download/source-separation-models/UVR-MDX-NET-Voc_FT.onnx"\nCONFIG = sherpa_onnx.OfflineSourceSeparationConfig(\n    model=sherpa_onnx.OfflineSourceSeparationModelConfig(\n        uvr=sherpa_onnx.OfflineSourceSeparationUvrModelConfig(\n            model="/content/UVR-MDX-NET-Voc_FT.onnx",\n        ),\n        num_threads=1,\n        debug=False,\n        provider="cuda",\n    )\n)\nif not CONFIG.validate():\n    raise RuntimeError("The exact sherpa-onnx CUDA separation configuration is invalid")\nSEPARATOR = sherpa_onnx.OfflineSourceSeparation(CONFIG)\n\nTOKEN = os.environ["LA_STUDIO_COLAB_SEPARATION_TOKEN"]\nROOT = Path("/content/la-studio-separation-jobs") / MODEL_ID\nROOT.mkdir(parents=True, exist_ok=True)\nMAX_UPLOAD_BYTES = 512 * 1024 * 1024\nMAX_AUDIO_SECONDS = 30 * 60\nARTIFACT_TTL_SECONDS = 1800\nALLOWED_CONTENT_TYPES = {\n    "audio/wav", "audio/x-wav", "audio/mpeg", "audio/mp4", "audio/webm",\n    "audio/ogg", "audio/flac", "video/mp4", "video/webm", "video/quicktime",\n    "video/x-matroska", "application/octet-stream",\n}\nALLOWED_EXTENSIONS = {".wav", ".mp3", ".m4a", ".mp4", ".webm", ".ogg", ".flac", ".mkv", ".mov", ".avi"}\nJOB_SLOTS = threading.BoundedSemaphore(1)\nJOB_LOCK = threading.Lock()\nJOBS = {}\n\ndef authorize(authorization: str | None) -> None:\n    if authorization != "Bearer " + TOKEN:\n        raise HTTPException(status_code=401, detail="invalid worker token")\n\ndef require_exact_model(requested: str) -> None:\n    if requested.strip().lower() != MODEL_ID:\n        raise HTTPException(\n            status_code=409,\n            detail=f"This worker loaded \'{MODEL_ID}\', but LA Studio requested \'{requested}\'. Open the notebook for the selected model.",\n        )\n\ndef media_duration_seconds(path: Path) -> float:\n    probe = subprocess.run(\n        ["ffprobe", "-v", "error", "-show_entries", "format=duration",\n         "-of", "default=nokey=1:noprint_wrappers=1", str(path)],\n        text=True, capture_output=True,\n    )\n    try:\n        duration = float(probe.stdout.strip())\n    except ValueError:\n        duration = 0.0\n    if probe.returncode != 0 or duration <= 0.0:\n        raise HTTPException(status_code=415, detail="media is unsupported or could not be decoded")\n    return duration\n\ndef update(job_id: str, **values) -> dict:\n    with JOB_LOCK:\n        job = dict(JOBS.get(job_id, {}))\n        job.update(values)\n        JOBS[job_id] = job\n        return job\n\ndef cleanup(job_id: str) -> None:\n    with JOB_LOCK:\n        job = JOBS.pop(job_id, None)\n    if job:\n        shutil.rmtree(job.get("directory", ""), ignore_errors=True)\n\ndef run_job(job_id: str, directory: Path, source: Path, output_format: str) -> None:\n    try:\n        update(job_id, status="running", progress=20, detail=f"{MODEL_NAME} is separating vocals on CUDA")\n        wav_path = directory / "source-44100-stereo.wav"\n        subprocess.run(\n            ["ffmpeg", "-y", "-v", "error", "-i", str(source), "-vn", "-acodec", "pcm_s16le",\n             "-ar", "44100", "-ac", "2", str(wav_path)],\n            check=True,\n        )\n        samples, sample_rate = sf.read(wav_path, dtype="float32", always_2d=True)\n        samples = np.ascontiguousarray(samples.T)\n        output = SEPARATOR.process(sample_rate=sample_rate, samples=samples)\n        if len(output.stems) != 2:\n            raise RuntimeError(f"expected two stems, received {len(output.stems)}")\n        suffix = ".wav" if output_format == "wav" else ".flac"\n        file_format = "WAV" if output_format == "wav" else "FLAC"\n        vocals = directory / ("vocals" + suffix)\n        background = directory / ("background" + suffix)\n        sf.write(vocals, np.asarray(output.stems[0].data).T, output.sample_rate,\n                 format=file_format, subtype="PCM_16")\n        sf.write(background, np.asarray(output.stems[1].data).T, output.sample_rate,\n                 format=file_format, subtype="PCM_16")\n        with JOB_LOCK:\n            cancelled = JOBS.get(job_id, {}).get("cancel_requested", False)\n        if cancelled:\n            vocals.unlink(missing_ok=True)\n            background.unlink(missing_ok=True)\n            update(job_id, status="cancelled", progress=0, detail="Separation cancelled")\n        else:\n            update(\n                job_id, status="ready", progress=100, detail="Separated stems are ready",\n                vocals=str(vocals), background=str(background),\n                artifact_format=output_format, artifacts_ready=True,\n            )\n    except Exception as error:\n        update(job_id, status="failed", progress=0, detail=f"{type(error).__name__}: {str(error)[:1800]}")\n    finally:\n        threading.Timer(ARTIFACT_TTL_SECONDS, cleanup, args=[job_id]).start()\n        JOB_SLOTS.release()\n\napp = FastAPI(title=f"LA Studio Voice Isolation - {MODEL_NAME}", docs_url=None, redoc_url=None, openapi_url=None)\n\n@app.get("/health")\n@app.get("/v1/health")\ndef health(authorization: str | None = Header(default=None)):\n    authorize(authorization)\n    return {\n        "status": "ready",\n        "ready": True,\n        "device": "cuda",\n        "model": MODEL_ID,\n        "variant": "fixed",\n        "upstream_model": UPSTREAM_MODEL,\n        "sherpa_onnx": sherpa_onnx.__version__,\n        "cpu_fallback": False,\n    }\n\n@app.get("/v1/capabilities")\ndef capabilities(authorization: str | None = Header(default=None)):\n    authorize(authorization)\n    return {\n        "contract_version": 1,\n        "device": "cuda",\n        "capabilities": [{\n            "id": "voice-isolation",\n            "models": [{\n                "id": MODEL_ID,\n                "name": MODEL_NAME,\n                "variant": "fixed",\n                "upstream_model": UPSTREAM_MODEL,\n                "artifact_url": ARTIFACT_URL,\n                "stems": ["vocals", "background"],\n                "formats": ["flac", "wav"],\n                "device": "cuda",\n                "loaded": True,\n            }],\n        }],\n    }\n\n@app.post("/v1/audio/separations")\nasync def create_separation(\n    file: UploadFile = File(...),\n    stems: str = Form("vocals,background"),\n    model: str = Form(...),\n    output_format: str = Form("flac"),\n    authorization: str | None = Header(default=None),\n):\n    authorize(authorization)\n    require_exact_model(model)\n    if stems != "vocals,background":\n        raise HTTPException(status_code=422, detail="this worker returns vocals and background stems")\n    output_format = output_format.strip().lower()\n    if output_format not in {"flac", "wav"}:\n        raise HTTPException(status_code=422, detail="output_format must be flac or wav")\n    suffix = Path(file.filename or "source.wav").suffix.lower() or ".wav"\n    if suffix not in ALLOWED_EXTENSIONS:\n        raise HTTPException(status_code=415, detail="unsupported media filename extension")\n    if file.content_type and file.content_type not in ALLOWED_CONTENT_TYPES:\n        raise HTTPException(status_code=415, detail="unsupported media MIME type")\n    if not JOB_SLOTS.acquire(blocking=False):\n        raise HTTPException(status_code=429, detail="the Colab separation worker is busy; retry shortly")\n    job_id = secrets.token_urlsafe(18)\n    directory = ROOT / job_id\n    directory.mkdir(parents=True, exist_ok=True)\n    source = directory / ("source" + suffix)\n    try:\n        with source.open("wb") as output:\n            while chunk := await file.read(1024 * 1024):\n                output.write(chunk)\n                if output.tell() > MAX_UPLOAD_BYTES:\n                    raise HTTPException(status_code=413, detail="media exceeds 512 MB upload limit")\n        if source.stat().st_size <= 0:\n            raise HTTPException(status_code=413, detail="media must not be empty")\n        if media_duration_seconds(source) > MAX_AUDIO_SECONDS:\n            raise HTTPException(status_code=413, detail="media exceeds the 30 minute duration limit")\n    except Exception:\n        shutil.rmtree(directory, ignore_errors=True)\n        JOB_SLOTS.release()\n        raise\n    finally:\n        await file.close()\n    update(\n        job_id, status="queued", progress=10, detail=f"Media uploaded; {MODEL_NAME} CUDA job is queued",\n        directory=str(directory), cancel_requested=False,\n    )\n    threading.Thread(target=run_job, args=(job_id, directory, source, output_format), daemon=True).start()\n    return {"job_id": job_id, "status": "queued", "progress": 10,\n            "artifact_format": output_format}\n\n@app.get("/v1/audio/separations/{job_id}")\ndef separation_status(job_id: str, authorization: str | None = Header(default=None)):\n    authorize(authorization)\n    with JOB_LOCK:\n        job = dict(JOBS.get(job_id, {}))\n    if not job:\n        raise HTTPException(status_code=404, detail="separation job not found")\n    return {key: job.get(key) for key in ("status", "progress", "detail", "artifact_format", "artifacts_ready") if key in job} | {"job_id": job_id}\n\n@app.get("/v1/audio/separations/{job_id}/artifacts/{stem}")\ndef artifact(job_id: str, stem: str, authorization: str | None = Header(default=None)):\n    authorize(authorization)\n    if stem not in {"vocals", "background"}:\n        raise HTTPException(status_code=404, detail="unknown stem")\n    with JOB_LOCK:\n        job = dict(JOBS.get(job_id, {}))\n    path = Path(job.get(stem, ""))\n    if job.get("status") != "ready" or not path.is_file():\n        raise HTTPException(status_code=409, detail="stem is not ready")\n    output_format = job.get("artifact_format", "wav")\n    media_type = "audio/wav" if output_format == "wav" else "audio/flac"\n    return FileResponse(path, media_type=media_type, filename=stem + "." + output_format)\n\n@app.delete("/v1/audio/separations/{job_id}")\ndef cancel_separation(job_id: str, authorization: str | None = Header(default=None)):\n    authorize(authorization)\n    with JOB_LOCK:\n        if job_id not in JOBS:\n            raise HTTPException(status_code=404, detail="separation job not found")\n        JOBS[job_id]["cancel_requested"] = True\n        JOBS[job_id]["status"] = "cancelling"\n    return {"job_id": job_id, "status": "cancelling"}\n', encoding='utf-8')
print('Worker source:', WORKER)


In [ ]:
MODEL_ID = 'sherpa-onnx-uvr-vocals-ft'
# LA Studio worker launch contract: launch-2026-08-06.1
import json
import os
import queue
import re
import secrets
import signal
import socket
import subprocess
import sys
import threading
import time
import urllib.error
import urllib.request
from pathlib import Path

CAPABILITY_LABEL = 'Voice Isolation'
MODEL_ID = 'sherpa-onnx-uvr-vocals-ft'
PORT = 3924
TOKEN_ENV = 'LA_STUDIO_COLAB_SEPARATION_TOKEN'
URL_ENV = 'LA_STUDIO_COLAB_SEPARATION_URL'
MODEL_ENV = 'LA_STUDIO_COLAB_SEPARATION_MODEL'
WORKER_LOG = Path('/content/la_studio_separation_worker.log')
WORKER_MODULE = 'la_studio_separation_worker'
WORKER_PYTHON = sys.executable
WORKER_PYTHON_ISOLATED = False
WORKER_ENVIRONMENT = {}
REQUIRES_CUDA = True
STARTUP_TIMEOUT_SECONDS = 20 * 60
TUNNEL_TIMEOUT_SECONDS = 90
TOKEN = secrets.token_urlsafe(32)


def port_is_occupied(port: int) -> bool:
    try:
        with socket.create_connection(("127.0.0.1", port), timeout=0.5):
            return True
    except OSError:
        return False


def process_cmdline(pid: int) -> str:
    """Read a Linux process command line without depending on psutil."""
    try:
        return Path(f"/proc/{pid}/cmdline").read_bytes().replace(b"\0", b" ").decode(
            "utf-8", errors="replace"
        ).strip()
    except (FileNotFoundError, PermissionError, ProcessLookupError):
        return ""


def all_processes():
    for entry in Path("/proc").iterdir():
        if not entry.name.isdigit():
            continue
        pid = int(entry.name)
        command = process_cmdline(pid)
        if command:
            yield pid, command


def listening_processes(port: int) -> dict[int, str]:
    """Return PIDs listening on a local TCP port via /proc socket ownership."""
    target_port = f"{port:04X}"
    socket_inodes = set()
    for table_name in ("/proc/net/tcp", "/proc/net/tcp6"):
        try:
            lines = Path(table_name).read_text(encoding="utf-8").splitlines()[1:]
        except FileNotFoundError:
            continue
        for line in lines:
            fields = line.split()
            if len(fields) < 10:
                continue
            local_address, state, inode = fields[1], fields[3], fields[9]
            if state == "0A" and local_address.rsplit(":", 1)[-1].upper() == target_port:
                socket_inodes.add(inode)
    if not socket_inodes:
        return {}

    listeners = {}
    for entry in Path("/proc").iterdir():
        if not entry.name.isdigit():
            continue
        try:
            descriptors = (entry / "fd").iterdir()
        except (FileNotFoundError, PermissionError):
            continue
        for descriptor in descriptors:
            try:
                target = os.readlink(descriptor)
            except (FileNotFoundError, PermissionError, OSError):
                continue
            match = re.fullmatch(r"socket:\\[(\\d+)\\]", target)
            if match and match.group(1) in socket_inodes:
                pid = int(entry.name)
                listeners[pid] = process_cmdline(pid)
                break
    return listeners


def stop_pid(pid: int) -> None:
    if pid == os.getpid():
        return
    try:
        os.kill(pid, signal.SIGTERM)
    except ProcessLookupError:
        return
    deadline = time.monotonic() + 10
    while time.monotonic() < deadline:
        try:
            os.kill(pid, 0)
        except ProcessLookupError:
            return
        time.sleep(0.2)
    try:
        os.kill(pid, signal.SIGKILL)
    except ProcessLookupError:
        pass


def reclaim_previous_la_studio_worker() -> None:
    """Stop only an older LA Studio worker/tunnel for this exact local port.

    Re-running a Colab cell keeps child processes alive.  The previous launch
    created a new token but aborted before it could replace the old worker,
    forcing users to destroy the whole GPU runtime.  We identify ownership by
    the exact generated module name and never terminate a foreign listener.
    """
    stopped = []
    for pid, command in listening_processes(PORT).items():
        if WORKER_MODULE in command and "uvicorn" in command:
            stop_pid(pid)
            stopped.append(f"worker PID {pid}")

    endpoint = f"http://127.0.0.1:{PORT}"
    for pid, command in all_processes():
        if ("cloudflared" in command and "tunnel" in command and endpoint in command):
            stop_pid(pid)
            stopped.append(f"tunnel PID {pid}")

    deadline = time.monotonic() + 12
    while port_is_occupied(PORT) and time.monotonic() < deadline:
        time.sleep(0.2)
    if stopped:
        print("Stopped previous LA Studio " + ", ".join(stopped) + ".")

    if port_is_occupied(PORT):
        listeners = listening_processes(PORT)
        foreign_pids = sorted(listeners) or ["unknown"]
        raise RuntimeError(
            f"Port {PORT} is occupied by a process that is not the previous LA Studio "
            f"{CAPABILITY_LABEL} worker (PID(s): {', '.join(map(str, foreign_pids))}). "
            "Choose a fresh Colab runtime rather than terminating an unrelated process."
        )


def worker_log_tail() -> str:
    try:
        return WORKER_LOG.read_text(encoding="utf-8", errors="replace")[-12000:]
    except FileNotFoundError:
        return "(worker log was not created)"


def stop_process(process) -> None:
    if process is None or process.poll() is not None:
        return
    process.terminate()
    try:
        process.wait(timeout=10)
    except subprocess.TimeoutExpired:
        process.kill()


reclaim_previous_la_studio_worker()

env = os.environ.copy()
env[TOKEN_ENV] = TOKEN
env["PYTHONUNBUFFERED"] = "1"
env.update(WORKER_ENVIRONMENT)
if WORKER_PYTHON_ISOLATED:
    # Do not let Colab's global site-packages or a notebook-level PYTHONPATH
    # bleed into a dedicated worker virtual environment.
    env.pop("PYTHONPATH", None)
    env["PYTHONNOUSERSITE"] = "1"
worker = None
tunnel = None

with WORKER_LOG.open("w", encoding="utf-8", buffering=1) as worker_output:
    worker = subprocess.Popen(
        [WORKER_PYTHON, "-m", "uvicorn", 'la_studio_separation_worker:app', "--host", "127.0.0.1", "--port", str(PORT)],
        cwd="/content",
        env=env,
        stdout=worker_output,
        stderr=subprocess.STDOUT,
    )
    worker_kind = "exact CUDA" if REQUIRES_CUDA else "dedicated Colab CPU"
    print(f"Starting {worker_kind} {CAPABILITY_LABEL} worker.")
    deadline = time.monotonic() + STARTUP_TIMEOUT_SECONDS
    last_error = "worker has not answered /health yet"
    next_report = time.monotonic()
    while time.monotonic() < deadline:
        exit_code = worker.poll()
        if exit_code is not None:
            raise RuntimeError(
                f"The exact-model {CAPABILITY_LABEL} worker exited before becoming ready (exit code {exit_code}).\n\n"
                "---- LA Studio worker log (last 12,000 characters) ----\n" + worker_log_tail()
            )
        try:
            request = urllib.request.Request(
                f"http://127.0.0.1:{PORT}/health",
                headers={"Authorization": "Bearer " + TOKEN},
            )
            with urllib.request.urlopen(request, timeout=10) as response:
                health = json.loads(response.read().decode("utf-8"))
            if (response.status == 200
                    and health.get("ready") is True
                    and str(health.get("device", "")).lower()
                        == ("cuda" if REQUIRES_CUDA else "colab-cpu")
                    and str(health.get("model", "")).strip().lower() == MODEL_ID
                    and health.get("cpu_fallback") is False):
                print(worker_kind.title() + " worker is ready:", health)
                break
            last_error = "unexpected /health response: " + json.dumps(health, ensure_ascii=False)
        except urllib.error.HTTPError as error:
            last_error = f"/health returned HTTP {error.code}: " + error.read().decode("utf-8", errors="replace")[:1000]
        except Exception as error:
            last_error = f"/health is not ready: {type(error).__name__}: {error}"
        if time.monotonic() >= next_report:
            print(f"Waiting for the {worker_kind} worker...", last_error)
            next_report = time.monotonic() + 30
        time.sleep(2)
    else:
        stop_process(worker)
        raise RuntimeError(
            f"The {worker_kind} {CAPABILITY_LABEL} worker did not become ready within "
            f"{STARTUP_TIMEOUT_SECONDS // 60} minutes. Last health-check result: {last_error}\n\n"
            "---- LA Studio worker log (last 12,000 characters) ----\n" + worker_log_tail()
        )


def cloudflared_ready() -> bool:
    try:
        return subprocess.run(
            ["cloudflared", "--version"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
            check=False,
        ).returncode == 0
    except OSError:
        return False


def ensure_cloudflared() -> None:
    if cloudflared_ready():
        return
    package_path = "/content/la-studio-cloudflared.deb"
    download = subprocess.run(
        [
            "curl", "--fail", "--location", "--retry", "4", "--retry-all-errors",
            "--output", package_path,
            "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb",
        ],
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        check=False,
    )
    if download.returncode != 0:
        detail = download.stdout[-1200:].strip() or "no download output"
        raise RuntimeError("Could not download cloudflared: " + detail)
    install = subprocess.run(
        ["dpkg", "-i", package_path], text=True, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, check=False,
    )
    if install.returncode != 0 or not cloudflared_ready():
        detail = install.stdout[-1200:].strip() or "no installation output"
        raise RuntimeError("Could not install cloudflared: " + detail)


ensure_cloudflared()
tunnel = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{PORT}", "--no-autoupdate"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
tunnel_lines = queue.Queue()


def collect_tunnel_output() -> None:
    assert tunnel.stdout is not None
    for line in tunnel.stdout:
        tunnel_lines.put(line)


threading.Thread(target=collect_tunnel_output, daemon=True).start()
public_url = ""
recent_tunnel_lines = []
deadline = time.monotonic() + TUNNEL_TIMEOUT_SECONDS
while time.monotonic() < deadline and not public_url:
    if tunnel.poll() is not None:
        break
    try:
        line = tunnel_lines.get(timeout=1)
    except queue.Empty:
        continue
    recent_tunnel_lines.append(line.rstrip())
    recent_tunnel_lines = recent_tunnel_lines[-10:]
    print(line, end="")
    match = re.search(r"https://[^\s\"']+\.trycloudflare\.com", line)
    if match:
        # The desktop Check Colab action is the authoritative public endpoint,
        # bearer-token, capability, and exact-model verification.
        public_url = match.group(0)

if not public_url:
    stop_process(tunnel)
    stop_process(worker)
    tail = "\n".join(recent_tunnel_lines) or "(no cloudflared output)"
    raise RuntimeError(
        f"cloudflared did not publish a trycloudflare URL within {TUNNEL_TIMEOUT_SECONDS} seconds.\n"
        "---- cloudflared output ----\n" + tail
    )

os.environ[URL_ENV] = public_url
os.environ[TOKEN_ENV] = TOKEN
os.environ[MODEL_ENV] = MODEL_ID
print("\nLA Studio exact-model Colab worker is ready")
print(URL_ENV + "=" + public_url)
print(TOKEN_ENV + "=" + TOKEN)
print(MODEL_ENV + "=" + MODEL_ID)
print("Click Check Colab in the matching LA Studio feature before running it.")
